# 04.1 — The architecture string dispatcher

MATLAB's pipeline selects a network from a **string**: `ModelName = 'GRU'`, `ClassifierName = 'Deep LSTM - Dropout 0.5'`. One config field, and a giant `switch` builds the right architecture. This project replaces that switch with a **registry** — the same functions-as-values pattern from [01.3](../01_python_for_matlab_users/01.3_functions_and_lambdas.ipynb), now at production scale, and the backbone of all of Module 04.

This notebook covers:

* The dispatch problem: string → constructed model.
* MATLAB's `switch` vs Python's registry, side by side.
* `register_encoder` / `build_encoder` and the decorator that populates them.
* Why a registry beats a switch: extension without modification.

**Prerequisites:** [01.3 functions](../01_python_for_matlab_users/01.3_functions_and_lambdas.ipynb), [02.6 nn.Module](../02_numpy_and_pytorch_basics/02.6_nn_module_vs_layergraph.ipynb).

## Section 1 — What MATLAB does

`PARAMETERS_cgg_constructNetworkArchitecture` (and its callers) branch on the model-name string:

```matlab
switch ModelName
    case 'GRU'
        layers = buildGRUEncoder(cfg);
    case 'LSTM'
        layers = buildLSTMEncoder(cfg);
    case 'Feedforward'
        layers = buildFeedforwardEncoder(cfg);
    case 'Convolutional'
        layers = buildConvEncoder(cfg);
    % ... ~47 architecture options across encoders + classifiers ...
    otherwise
        error('Unknown ModelName: %s', ModelName);
end
```

Every architecture is a `case`. Adding one means editing this central function — and every place that switches on the same string. The Python registry inverts that: architectures *register themselves*, and the dispatcher never changes.

## Section 2 — The Python concepts you need

### 2.1 — A registry from scratch

A registry is a dict mapping names to builder functions, plus a decorator that fills it. Twelve lines reproduce the whole idea:

In [ ]:
_REGISTRY = {}                       # name → builder function

def register(name):
    """Decorator: store the decorated function under `name`."""
    def wrap(fn):
        _REGISTRY[name] = fn
        return fn
    return wrap

def build(name, cfg):
    if name not in _REGISTRY:
        raise KeyError(f"Unknown: {name!r}. Known: {sorted(_REGISTRY)}")
    return _REGISTRY[name](cfg)

# Architectures register THEMSELVES — no central switch to edit:
@register("tiny")
def _build_tiny(cfg):
    import torch.nn as nn
    return nn.Linear(cfg["in"], cfg["out"])

@register("deep")
def _build_deep(cfg):
    import torch.nn as nn
    return nn.Sequential(nn.Linear(cfg["in"], 16), nn.ReLU(), nn.Linear(16, cfg["out"]))

print("registered:", sorted(_REGISTRY))
print(build("tiny", {"in": 4, "out": 2}))

The decorator (`@register("tiny")`) runs at import time — the moment Python reads the `def`, the function lands in `_REGISTRY`. `build` is a pure lookup that never mentions any specific architecture. Compare the two designs:

| | MATLAB `switch` | Python registry |
|---|---|---|
| add an architecture | edit the central `switch` (and every switch on that string) | add a `@register(...)` in its own file |
| the dispatcher | grows with every case | never changes |
| discovery | read the whole switch | `list(_REGISTRY)` |
| coupling | dispatcher knows every architecture | dispatcher knows none |

This is the **open/closed principle** — open for extension (new registrations), closed for modification (the dispatcher is frozen). It's why the codebase can carry 7 encoders + 3 classifiers with a dispatcher that's a dozen lines.

### 2.2 — The production registry, live

In [ ]:
import neural_data_decoding.models          # importing the package runs every @register
from neural_data_decoding.models.registry import list_encoders, list_classifiers

print("encoders   :", list_encoders())
print("classifiers:", list_classifiers())

Those names are exactly the MATLAB `ModelName` / `ClassifierName` strings — the config carries the same value, so a MATLAB config field maps to a Python build with no translation. `import neural_data_decoding.models` is what populates the registry: each encoder/classifier module runs its `@register_encoder(...)` decorator on import (the reason the package `__init__` imports them all).

### 2.3 — Building from a name

In [ ]:
from neural_data_decoding.models.registry import build_encoder
import torch

# The string "GRU" plus a config dict → a constructed, ready-to-run module:
encoder = build_encoder("GRU", {
    "in_features": 8, "hidden_sizes": [16, 4],
    "transform": "GRU", "dropout": 0.0,
    "want_normalization": False, "activation": "",
})
print(type(encoder).__name__, "—", sum(p.numel() for p in encoder.parameters()), "params")

x = torch.randn(2, 5, 8)          # (batch, sequence, features)
print("forward:", tuple(encoder(x).shape))

One call turns a string into a working `nn.Module`. The CLI does exactly this — reads `cfg.model_name`, calls `build_encoder`, and never contains an `if model_name == "GRU"` anywhere. Swapping architectures in a sweep ([Module 09](../README.md)) is a config change, not a code change.

## Section 3 — The `neural_data_decoding` implementation

The real registry — same shape as your from-scratch version in §2.1, with error handling and type hints:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/models/registry.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if line.startswith("def register_encoder"))
for k in range(i, min(i + 22, len(src))):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* `register_encoder` and `register_classifier` are two registries (`_ENCODER_REGISTRY`, `_CLASSIFIER_REGISTRY`) built by the same private `_register` helper — the DRY version of §2.1's `register`.
* `build_encoder` raises a `KeyError` **listing the known names** on a miss — a good error message ([01.6](../01_python_for_matlab_users/01.6_error_handling.ipynb)) turns "unknown GRU" into "you meant one of these seven."
* Each encoder module (`encoder.py`, `conv_encoder.py`, …) decorates its builder — so the registry's contents are distributed across the files that define them, never centralized.

In [ ]:
# The KeyError with its helpful listing — trigger it:
try:
    build_encoder("Convolutionl", {})       # typo
except KeyError as e:
    print(e)

## Section 4 — Hands-on exercises

### Exercise 4.1 — register your own

Add a `"Passthrough"` encoder to a *local* toy registry (reusing §2.1's `register`/`build`) that returns `nn.Identity()`, then build it. (We use the toy registry, not the production one, to avoid polluting it.)

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
import torch.nn as nn

@register("Passthrough")
def _build_passthrough(cfg):
    return nn.Identity()

model = build("Passthrough", {})
x = torch.randn(3, 5)
print("Identity preserves input:", torch.equal(model(x), x))

### Exercise 4.2 — the config carries the choice

Given a list of model names, build each and report its parameter count — the loop a sweep runs. (Use a minimal cfg that suits the simple encoders: `GRU`, `LSTM`, `Feedforward`.)

In [ ]:
# Reveal:
base_cfg = {"in_features": 8, "hidden_sizes": [16, 4], "dropout": 0.0,
            "want_normalization": False, "activation": ""}
for name in ["GRU", "LSTM", "Feedforward"]:
    m = build_encoder(name, {**base_cfg, "transform": name})
    print(f"  {name:12} → {sum(p.numel() for p in m.parameters()):5d} params")

## Section 5 — Common errors

### `KeyError: No encoder registered under '...'`

Typo in the model name, or the module defining it was never imported (so its `@register` never ran). `import neural_data_decoding.models` triggers all registrations; `list_encoders()` shows what's actually available.

### The architecture builds but forwards wrong / errors on shape

The registry only guarantees "name → some module." Whether that module fits your data is separate — encoders expect `(batch, sequence, features)` with `in_features` matching the last axis ([02.2](../02_numpy_and_pytorch_basics/02.2_array_axis_conventions.ipynb)). Mismatch surfaces at the first forward.

### Registering the same name twice

The later `@register` silently overwrites the earlier — a footgun if two files claim one name. The production `_register` can be made to reject duplicates; check its body if a registration mysteriously "doesn't take."

### `@register` seems to do nothing

Decorators run when the module is *imported*. If nobody imports the module, its registrations never happen. This is why the package `__init__` imports every architecture module eagerly.

### Adding a `case` mentality

Coming from the switch, the instinct is to find "the place that lists all architectures" and add yours there. There isn't one — add a `@register(...)`-decorated builder in a new (or the relevant) module, and the dispatcher picks it up automatically.

## Section 6 — Further reading

- [Design Patterns: registry / factory](https://python-patterns.guide/gang-of-four/factory-method/) — the pattern in the abstract.
- [`src/neural_data_decoding/models/registry.py`](../../src/neural_data_decoding/models/registry.py) — the full production registry, docstrings first.
- [`architecture_registry.py`](../../src/neural_data_decoding/models/architecture_registry.py) — the CC.1 spec registry that layers named *architecture specs* on top of the builder registry.

Next up: **[04.2 building a simple encoder](04.2_building_a_simple_encoder.ipynb)** — inside one of the builders the registry dispatches to.